# Unidad 6 · Colab 3 de 3
## Despliegue en PaaS: Render, Railway y Hugging Face Spaces

**Objetivos de este notebook**

- Diferenciar PaaS, IaaS y SaaS.
- Preparar un proyecto (variables de entorno, puerto dinámico) para ser desplegado.
- Desplegar una API/web app en **Render**, **Railway** y **Hugging Face Spaces**.
- Conectar el despliegue automático con el workflow de CI del Colab 1 (integración continua + despliegue continuo).
- Resolver problemas comunes de despliegue.

> **Nivel:** intermedio. Este notebook asume que ya tenés la API dockerizada del Colab 2 y el workflow de CI del Colab 1.

---

## 1. PaaS vs. IaaS vs. SaaS

| Modelo | Qué administrás vos | Qué administra el proveedor | Ejemplos |
|---|---|---|---|
| **IaaS** | SO, runtime, app | Hardware, red, virtualización | AWS EC2, Google Compute Engine |
| **PaaS** | Solo tu código y configuración | SO, runtime, escalado, infraestructura | Render, Railway, Hugging Face Spaces |
| **SaaS** | Nada, solo lo usás | Todo | Gmail, Notion |

Un PaaS es ideal para esta unidad: conectás tu repositorio de GitHub y la plataforma se encarga de construir y correr tu app.

## 2. Preparar el proyecto para el despliegue

Las plataformas PaaS gratuitas asignan el puerto de forma dinámica mediante la variable de entorno `PORT`, así que tu app **no puede tener el puerto hardcodeado**. También necesitás definir tus dependencias y tus variables de entorno (secretos) de forma explícita.

```python
# main.py (FastAPI + uvicorn)
import os
import uvicorn

if __name__ == '__main__':
    port = int(os.environ.get('PORT', 8000))
    uvicorn.run('app.main:app', host='0.0.0.0', port=port)
```

Checklist antes de desplegar:

- [ ] `requirements.txt` (o `package.json`) actualizado y probado
- [ ] El puerto se lee desde `os.environ.get('PORT', ...)`
- [ ] Variables de entorno documentadas en `.env.example` (sin valores reales)
- [ ] `.gitignore` excluye `.env`
- [ ] La app corre localmente con `docker run` antes de desplegar (Colab 2)

### Ejercicio 1 — Leer el puerto desde una variable de entorno

Este código de una API en Flask tiene el puerto hardcodeado. Reescribilo para que lea el puerto desde la variable de entorno `PORT` (con `5000` como valor por defecto).

```python
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
```

<details>
<summary>💡 Ver solución</summary>

```python
import os

if __name__ == '__main__':
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port)
```

</details>

## 3. Despliegue en Render

Pasos:

1. Crear cuenta en [render.com](https://render.com) y conectar tu cuenta de GitHub.
2. **New +** → **Web Service** → seleccionar el repositorio.
3. Elegir entre **Docker** (usa tu `Dockerfile` del Colab 2) o runtime nativo (Render detecta Python/Node y define build/start command).
4. Configurar **Build Command** y **Start Command** (si no usás Docker), y las **Environment Variables** (equivalentes a tu `.env`).
5. Deploy: Render construye y publica tu app en una URL pública `https://tu-app.onrender.com`.
6. **Auto-deploy** queda activado por defecto: cada `push` a la rama configurada dispara un nuevo deploy.

Documentación oficial: [Tu primer deploy en Render](https://render.com/docs/your-first-deploy) · [Web Services](https://render.com/docs/web-services) · [Conectar GitHub](https://render.com/docs/github) · [Deploy de una app FastAPI](https://render.com/docs/deploy-fastapi)

> En el free tier, el servicio *duerme* tras un período de inactividad y el primer request luego de eso tarda más (cold start).

### Ejercicio 2 — Build y Start command en Render

Tu proyecto es una API **FastAPI** que corre con `uvicorn app.main:app`. Completá el Build Command y el Start Command que configurarías en Render (sin usar Docker).

- Build Command: `(completá acá)`
- Start Command: `(completá acá)`

<details>
<summary>💡 Ver solución</summary>

- Build Command: `pip install -r requirements.txt`
- Start Command: `uvicorn app.main:app --host 0.0.0.0 --port $PORT`

</details>

## 4. Despliegue en Railway

Pasos:

1. Crear cuenta en [railway.com](https://railway.com) y conectar GitHub.
2. **New Project** → **Deploy from GitHub repo** → elegir el repositorio.
3. Railway detecta automáticamente el `Dockerfile` (si existe) o usa Nixpacks para construir la imagen sin que escribas uno.
4. Configurar variables de entorno en la pestaña **Variables**.
5. Railway asigna un dominio público (`*.up.railway.app`) y **redeploya automáticamente en cada push** a la rama conectada.

Documentación oficial: [Quick Start](https://docs.railway.com/quick-start) · [GitHub Autodeploys](https://docs.railway.com/deployments/github-autodeploys) · [Build & Deploy](https://docs.railway.com/build-deploy)

> Tip: con **Controlling GitHub Autodeploys**, Railway puede esperar a que tu workflow de GitHub Actions termine en verde antes de desplegar — así conectás CI con CD sin escribir nada extra.

### Ejercicio 3 — Comparar plataformas

Completá la tabla comparando Render, Railway y Hugging Face Spaces según lo que investigaste (o usando los links de este notebook).

| | Render | Railway | HF Spaces |
|---|---|---|---|
| ¿Soporta Docker? | | | |
| ¿Duerme en el free tier? | | | |
| ¿Ideal para APIs custom? | | | |
| ¿Ideal para demos de ML? | | | |

<details>
<summary>💡 Ver solución</summary>

| | Render | Railway | HF Spaces |
|---|---|---|---|
| ¿Soporta Docker? | Sí | Sí (o Nixpacks sin Dockerfile) | Sí (Space tipo Docker) |
| ¿Duerme en el free tier? | Sí, tras inactividad | Uso limitado por créditos/mes | No es el foco; pensado para demos |
| ¿Ideal para APIs custom? | Sí | Sí | Posible, pero no es su caso de uso principal |
| ¿Ideal para demos de ML? | Aceptable | Aceptable | Sí, es su especialidad (Gradio/Streamlit) |

</details>

## 5. Despliegue en Hugging Face Spaces

Hugging Face Spaces está pensado para publicar demos, pero admite **Spaces tipo Docker** para correr cualquier app (incluida una API custom), no solo Gradio/Streamlit.

Pasos:

1. Crear cuenta en [huggingface.co](https://huggingface.co) → **New Space**.
2. Elegir SDK: **Docker** (para tu API custom) o **Gradio/Streamlit** (para una demo interactiva).
3. Subir tu `Dockerfile` (o conectar el repo con GitHub Actions para sincronizarlo con el Space).
4. **Importante:** un Space tipo Docker debe exponer el puerto **7860** (fijo, no configurable por variable de entorno).

Documentación oficial: [Spaces Overview](https://huggingface.co/docs/hub/en/spaces-overview) · [Docker Spaces](https://huggingface.co/docs/hub/en/spaces-sdks-docker) · [Tu primer Docker Space](https://huggingface.co/docs/hub/en/spaces-sdks-docker-first-demo)

### Ejercicio 4 — Dockerfile para Hugging Face Spaces

Adaptá el `Dockerfile` de tu API de FastAPI (Colab 2) para que funcione en un Space tipo Docker, que exige exponer el puerto `7860`.

```dockerfile
(completá acá)
```

<details>
<summary>💡 Ver solución</summary>

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860
CMD uvicorn app.main:app --host 0.0.0.0 --port 7860
```

</details>

## 6. Conectar CI (Colab 1) con el despliegue

El workflow `.github/workflows/ci.yml` que escribiste en el Colab 1 corre los tests en cada push. El **auto-deploy** de Render/Railway/HF Spaces cumple el rol de **CD (Continuous Deployment)**: si el código en `main` cambia, se despliega solo.

Pipeline completo:

```text
push a una rama → Pull Request → GitHub Actions corre los tests
   ↓ (si pasan y se aprueba el PR)
merge a main
   ↓
Render/Railway/HF Spaces detecta el push a main
   ↓
build automático de la imagen/app
   ↓
deploy en la URL pública
```

Esto es integración continua (CI) + despliegue continuo (CD), sin necesidad de infraestructura propia.

### Ejercicio 5 — Diseñar el pipeline

Describí, en tus palabras, el pipeline completo que usarías para tu API: desde que hacés un cambio en tu editor hasta que ese cambio está disponible en la URL pública. Mencioná en qué paso interviene GitHub Actions y en qué paso interviene la plataforma PaaS que elegiste.

`(completá acá)`

<details>
<summary>💡 Ver solución (ejemplo)</summary>

Creo una rama `feature/nuevo-endpoint`, hago el cambio y abro un PR. GitHub Actions corre automáticamente los tests definidos en `ci.yml`. Si pasan y superan la revisión, mergeo el PR a `main`. Render detecta el push a `main`, reconstruye la imagen con el `Dockerfile` y despliega la nueva versión en `https://mi-api.onrender.com`, sin que yo tenga que hacer nada manualmente.

</details>

## 7. Problemas comunes al desplegar

| Síntoma | Causa probable | Solución |
|---|---|---|
| El build falla por falta de una variable | No configuraste una env var que el código espera | Revisar `.env.example` y cargarla en el panel de la plataforma |
| La app no responde / *port scan timeout* | El código no lee `PORT` del entorno, o escucha en `localhost` en vez de `0.0.0.0` | Usar `host=0.0.0.0` y `port=os.environ.get('PORT')` |
| Primer request muy lento | *Cold start* tras dormir por inactividad (free tier) | Esperado en free tier; considerar un ping periódico o plan pago si es crítico |
| Deploy exitoso pero 404/500 | Falta una dependencia en `requirements.txt`, o el Start Command apunta al módulo equivocado | Revisar logs del deploy en el panel de la plataforma |

## Mini-proyecto final integrador

1. Elegí **una** plataforma (Render, Railway o HF Spaces) para desplegar la API dockerizada del Colab 2.
2. Configurá las variables de entorno necesarias en el panel de la plataforma.
3. Verificá que tu workflow de CI (Colab 1) siga corriendo en cada push.
4. Hacé un cambio, mergealo a `main` y confirmá que el deploy automático se dispara.
5. Probá la URL pública resultante.

**Entregable:**

- Link al repositorio de GitHub (con el workflow de Actions visible y en verde).
- URL pública de la app desplegada.
- Captura del panel de la plataforma mostrando el deploy exitoso.

## Autoevaluación

- [ ] El repositorio tiene una estructura clara (README, .gitignore, tests, workflows)
- [ ] Usé una rama y un Pull Request para al menos un cambio
- [ ] El proyecto tiene un `Dockerfile` funcional y un `.dockerignore`
- [ ] El workflow de GitHub Actions corre los tests automáticamente
- [ ] La app está desplegada y accesible en una URL pública
- [ ] El despliegue se actualiza automáticamente al hacer push a `main`

---

**Fin de la Unidad 6.** Con estos tres notebooks recorriste el ciclo completo: organizar el repo, contenedorizar la app y desplegarla con integración y despliegue continuos.